# Are Vision Transformers more robust than CNNs? A small, honest experiment

**Question:** after fine-tuning a small Vision Transformer (**DeiT-Tiny**) and a small CNN (**ResNet-18**) on the same dataset, how does each one's accuracy change when test images get worse (fog, blur, low light, noise)? And how fast is each model at inference?

**What this notebook does**
1. Downloads a small dataset (Oxford-IIIT Pets, 37 cat/dog breeds).
2. Fine-tunes both pretrained models with identical settings.
3. Evaluates them on clean images and on synthetically degraded images (Albumentations), at 5 severity levels each.
4. Plots accuracy vs. severity.
5. Measures inference latency.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.

**Tip:** the first time, leave `QUICK_TEST = True` (below). It runs a tiny version in a few minutes so you can check nothing is broken. Then set it to `False` and run everything again for the real numbers.

> This notebook contains **no results**. Every number you see will come from your own run.

## Step 0: Install libraries

- `timm` gives us the pretrained DeiT-Tiny.
- `albumentations` applies the image degradations. We pin it to version 1.x because the function arguments used below match that version (version 2 renamed some of them).

In [ ]:
!pip install -q timm "albumentations<2"

## Step 1: Imports and settings

All the knobs you might want to change are in one place here.

In [ ]:
import os, random, time, copy
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"   # stop albumentations from checking for updates

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import timm
import albumentations as A

print("torch", torch.__version__, "| timm", timm.__version__, "| albumentations", A.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU found. Training will be very slow. Use Runtime -> Change runtime type.")

# ----------------------------- settings you can change -----------------------------
QUICK_TEST = True        # True = tiny run to check the code works. False = the real experiment.
SEED = 42                # random seed, so runs are repeatable (approximately)
IMG_SIZE = 224           # both models expect 224x224 images
BATCH_SIZE = 64
LEARNING_RATE = 3e-4     # same for both models for fairness; NOT tuned
WEIGHT_DECAY = 0.01
SEVERITIES = [1, 2, 3, 4, 5]   # 1 = mild, 5 = strong

if QUICK_TEST:
    EPOCHS, NUM_EVAL_IMAGES = 1, 100
else:
    EPOCHS, NUM_EVAL_IMAGES = 5, 500
# --------------------------------------------------------------------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Step 2: Data

We use **Oxford-IIIT Pets**: photos of 37 cat and dog breeds. It is small enough for Colab but still a real (fine-grained) classification task.

- `trainval` (3,680 images) is split into **train (85%)** and **validation (15%)**. We use validation only to pick the best epoch.
- `test` (3,669 images) is kept **untouched** until the final evaluation. This is important: if you choose settings by looking at test accuracy, your reported numbers are too optimistic.

Training images get light augmentation (random crop and horizontal flip) so the models see slight variety. Both models use the standard ImageNet normalization, because both were pretrained on ImageNet.

In [ ]:
# Download the dataset (about 800 MB, takes a minute or two the first time)
trainval_raw = datasets.OxfordIIITPet(root="data", split="trainval", target_types="category", download=True)
test_raw     = datasets.OxfordIIITPet(root="data", split="test",     target_types="category", download=True)
NUM_CLASSES = len(trainval_raw.classes)
print("Number of classes:", NUM_CLASSES, "| trainval images:", len(trainval_raw), "| test images:", len(test_raw))

# Standard ImageNet normalization (mean/std per color channel)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
normalize = transforms.Normalize(mean=MEAN, std=STD)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),   # random crop, then resize to 224x224
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize,
])

# Split trainval into train / validation using a fixed random permutation
perm = np.random.RandomState(SEED).permutation(len(trainval_raw))
n_val = int(0.15 * len(perm))
val_idx, train_idx = perm[:n_val].tolist(), perm[n_val:].tolist()

# Two copies of the dataset: one with training augmentation, one without.
train_ds = Subset(datasets.OxfordIIITPet(root="data", split="trainval", target_types="category", transform=train_tf, download=True), train_idx)
val_ds   = Subset(datasets.OxfordIIITPet(root="data", split="trainval", target_types="category", transform=eval_tf,  download=True), val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("train:", len(train_ds), "| val:", len(val_ds))

### Test images as arrays

For the degradation experiment we need the raw pixels, because Albumentations works on NumPy arrays (height x width x 3, values 0-255).
We take a fixed random subset of the test set (`NUM_EVAL_IMAGES`), resize each image to 224x224, and keep it in memory.
Every model and every degradation level will use **exactly the same images**, so comparisons are fair.

In [ ]:
rng = np.random.RandomState(SEED)
eval_idx = rng.choice(len(test_raw), size=min(NUM_EVAL_IMAGES, len(test_raw)), replace=False)

clean_images = np.zeros((len(eval_idx), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
eval_labels  = np.zeros(len(eval_idx), dtype=np.int64)
for i, idx in enumerate(eval_idx):
    img, label = test_raw[int(idx)]                                   # img is a PIL image
    clean_images[i] = np.array(img.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    eval_labels[i] = label

print("Evaluation set:", clean_images.shape, clean_images.dtype)

## Step 3: The two models

**Fine-tuning** means starting from a network that already learned general visual features on ImageNet (1.3M images) and continuing to train it on our small dataset. We replace the last layer (the "classifier head", which originally predicted 1000 ImageNet classes) with a new one that predicts our 37 breeds, then train *all* the weights.

- **ResNet-18** is a convolutional neural network (CNN). Convolutions look at small local patches and build up from edges to shapes to objects.
- **DeiT-Tiny** is a Vision Transformer. It cuts the image into 16x16 patches, treats each patch like a "word", and uses self-attention so every patch can look at every other patch.

Parameter counts are printed so you can see the models differ in size. Keep that in mind when you interpret results.

In [ ]:
def make_resnet18():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)   # pretrained on ImageNet
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)                      # new classifier head
    return m

def make_deit_tiny():
    # timm automatically swaps in a new head when num_classes differs from 1000
    return timm.create_model("deit_tiny_patch16_224", pretrained=True, num_classes=NUM_CLASSES)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print("ResNet-18 parameters:", f"{count_params(make_resnet18()):,}")
print("DeiT-Tiny parameters:", f"{count_params(make_deit_tiny()):,}")

## Step 4: Fine-tuning

Both models get **identical** training settings: AdamW optimizer, the same learning rate, a cosine learning-rate schedule (the rate smoothly decays to 0), and the same number of epochs.
I did not tune anything per model. Tuning one model more than the other would make the comparison unfair; if you do tune, tune both the same amount.

After each epoch we measure accuracy on the validation set and remember the best weights.

In [ ]:
@torch.inference_mode()
def accuracy_on_loader(model, loader):
    # Accuracy = fraction of images where the highest-scoring class is the correct one.
    model.eval()                       # turn off training-only behavior (e.g. dropout)
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total


def train_model(model, name):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))

    best_val, best_state = -1.0, None
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        running_loss, n_seen = 0.0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()                       # clear old gradients
            loss = criterion(model(images), labels)     # forward pass + loss
            loss.backward()                             # compute gradients
            optimizer.step()                            # update weights
            scheduler.step()                            # update learning rate
            running_loss += loss.item() * images.size(0)
            n_seen += images.size(0)

        val_acc = accuracy_on_loader(model, val_loader)
        print(f"[{name}] epoch {epoch+1}/{EPOCHS} | train loss {running_loss/n_seen:.4f} | val acc {val_acc:.4f}")
        if val_acc > best_val:
            best_val, best_state = val_acc, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)                   # go back to the best epoch
    print(f"[{name}] done in {time.time()-start:.0f}s | best val acc {best_val:.4f}\n")
    return model

In [ ]:
# Re-seeding before each model keeps the setup comparable (same data order logic, same seed).
set_seed(SEED)
resnet = train_model(make_resnet18(), "ResNet-18")

set_seed(SEED)
deit = train_model(make_deit_tiny(), "DeiT-Tiny")

trained_models = {"ResNet-18": resnet, "DeiT-Tiny": deit}

## Step 5: Degrading the test images with Albumentations

We simulate four common real-world problems. Each has 5 severity levels:

| Degradation | What it imitates | Albumentations transform | What "severity" controls |
|---|---|---|---|
| **Fog** | haze, bad weather | `RandomFog` | fog density |
| **Blur** | out-of-focus or shaky camera | `GaussianBlur` | blur width (sigma, in pixels) |
| **Low light** | dusk, underexposure | `RandomBrightnessContrast` (negative brightness) | how much brightness is removed |
| **Noise** | sensor noise in dim scenes | `GaussNoise` | noise standard deviation (in 0-255 pixel units) |

**Important caveats**
- The severity values are **my own choices**, not calibrated to any benchmark. They are meant to go from "mild" to "clearly damaged". Look at the preview below and adjust the lists if they look too weak or too strong.
- Fog and noise are random. We seed the random generators so that each (degradation, severity) setting is reproducible, and we create each degraded set **once** and give the same degraded images to both models.
- For a standardized benchmark, look into ImageNet-C. This notebook is a small, transparent version of that idea.

In [ ]:
# Strength of each degradation at severity 1..5 (my own choices -- edit freely)
FOG_LEVELS   = [0.2, 0.4, 0.6, 0.8, 1.0]      # fog coefficient (0 = none, 1 = max)
BLUR_SIGMAS  = [1.0, 2.0, 3.0, 4.0, 6.0]      # Gaussian blur sigma in pixels
DARKNESS     = [0.2, 0.4, 0.6, 0.75, 0.9]     # fraction of brightness removed
NOISE_STDS   = [5, 10, 20, 30, 50]            # noise standard deviation (pixel values 0-255)

DEGRADATION_NAMES = ["fog", "blur", "low_light", "noise"]

def make_transform(kind, severity):
    # Returns an Albumentations transform for one degradation at one severity (1..5).
    i = severity - 1
    if kind == "fog":
        t = A.RandomFog(fog_coef_lower=FOG_LEVELS[i], fog_coef_upper=FOG_LEVELS[i], alpha_coef=0.08, p=1.0)
    elif kind == "blur":
        sigma = BLUR_SIGMAS[i]
        ksize = int(4 * sigma) + 1                       # kernel size must be odd (always is here)
        t = A.GaussianBlur(blur_limit=(ksize, ksize), sigma_limit=(sigma, sigma), p=1.0)
    elif kind == "low_light":
        t = A.RandomBrightnessContrast(brightness_limit=(-DARKNESS[i], -DARKNESS[i]), contrast_limit=(0, 0), p=1.0)
    elif kind == "noise":
        var = float(NOISE_STDS[i]) ** 2                  # this version of GaussNoise takes variance = std^2
        t = A.GaussNoise(var_limit=(var, var), p=1.0)
    else:
        raise ValueError(kind)
    return A.Compose([t])


def degrade_images(images, kind, severity, seed=SEED):
    # Apply one degradation to every image in a (N, H, W, 3) uint8 array.
    random.seed(seed); np.random.seed(seed)              # make the random fog/noise reproducible
    aug = make_transform(kind, severity)
    out = np.empty_like(images)
    for i in range(len(images)):
        out[i] = aug(image=images[i])["image"]
    return out

### Look at the degraded images (do this before trusting any numbers!)

Each row is one degradation; columns go from clean to severity 5. If a level looks identical to clean, or the image is completely destroyed at level 2, edit the level lists above.

In [ ]:
sample = clean_images[0]
fig, axes = plt.subplots(len(DEGRADATION_NAMES), 6, figsize=(14, 9))
for r, kind in enumerate(DEGRADATION_NAMES):
    axes[r, 0].imshow(sample)
    axes[r, 0].set_title("clean", fontsize=9)
    for s in SEVERITIES:
        degraded = degrade_images(sample[None], kind, s)[0]      # sample[None] adds a batch dimension
        axes[r, s].imshow(degraded)
        axes[r, s].set_title(f"{kind}, level {s}", fontsize=9)
    for ax in axes[r]:
        ax.axis("off")
plt.tight_layout()
plt.savefig("degradation_examples.png", dpi=150)
plt.show()

## Step 6: Evaluate on clean and degraded images

For every (degradation, severity) pair we:
1. create the degraded copy of the evaluation images (once),
2. run **both** models on it,
3. record accuracy.

The models are **not retrained or adapted** to the degradations. They only ever saw clean (lightly augmented) training images, so this measures how well their learned features survive a distribution shift.

In [ ]:
mean_t = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)
std_t  = torch.tensor(STD,  device=device).view(1, 3, 1, 1)

@torch.inference_mode()
def accuracy_on_array(model, images, labels, batch_size=100):
    # images: (N, H, W, 3) uint8 array. We convert to normalized float tensors in batches.
    model.eval()
    correct = 0
    for start in range(0, len(images), batch_size):
        batch = torch.from_numpy(images[start:start + batch_size]).to(device)   # uint8, (B, H, W, 3)
        batch = batch.permute(0, 3, 1, 2).float() / 255.0                       # -> (B, 3, H, W), range 0-1
        batch = (batch - mean_t) / std_t                                        # same normalization as training
        preds = model(batch).argmax(dim=1).cpu().numpy()
        correct += int((preds == labels[start:start + batch_size]).sum())
    return correct / len(images)


records = []

# Clean images (severity 0)
for name, model in trained_models.items():
    acc = accuracy_on_array(model, clean_images, eval_labels)
    records.append(dict(model=name, degradation="clean", severity=0, accuracy=acc))
    print(f"{name:10s} | clean | accuracy {acc:.4f}")

# Degraded images
for kind in DEGRADATION_NAMES:
    for sev in SEVERITIES:
        degraded = degrade_images(clean_images, kind, sev)      # same degraded images for both models
        for name, model in trained_models.items():
            acc = accuracy_on_array(model, degraded, eval_labels)
            records.append(dict(model=name, degradation=kind, severity=sev, accuracy=acc))
            print(f"{name:10s} | {kind:9s} level {sev} | accuracy {acc:.4f}")

results = pd.DataFrame(records)

# Accuracy relative to each model's own clean accuracy (helps because clean accuracies differ)
clean_acc = results[results.degradation == "clean"].set_index("model")["accuracy"]
results["relative_to_clean"] = results.apply(lambda r: r["accuracy"] / clean_acc[r["model"]], axis=1)

results.to_csv("results_accuracy.csv", index=False)
print("\nSaved results_accuracy.csv")

In [ ]:
# A compact table: rows = degradation + severity, columns = model
table = results[results.degradation != "clean"].pivot_table(
    index=["degradation", "severity"], columns="model", values="accuracy")
display(table.round(3))

## Step 7: Plot accuracy vs. degradation severity

One panel per degradation. Severity 0 on the x-axis is the clean image.

The error bars are an approximate **95% confidence interval** for accuracy, computed as 1.96 x sqrt(p(1-p)/n), where p is the accuracy and n is the number of evaluation images. It tells you how much the number could wobble just from using a limited sample of test images. If two models' error bars overlap heavily, be cautious about claiming one is better.

In [ ]:
n = len(eval_labels)
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)

for ax, kind in zip(axes, DEGRADATION_NAMES):
    for name in trained_models:
        c = results[(results.model == name) & (results.degradation == "clean")]["accuracy"].iloc[0]
        sub = results[(results.model == name) & (results.degradation == kind)].sort_values("severity")
        x = np.array([0] + sub["severity"].tolist())
        y = np.array([c] + sub["accuracy"].tolist())
        ci = 1.96 * np.sqrt(y * (1 - y) / n)
        ax.errorbar(x, y, yerr=ci, marker="o", capsize=3, label=name)
    ax.set_title(kind.replace("_", " ").title())
    ax.set_xlabel("Severity (0 = clean)")
    ax.set_xticks([0] + SEVERITIES)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Accuracy")
axes[0].legend()
plt.suptitle(f"Accuracy vs. degradation severity (n = {n} test images)")
plt.tight_layout()
plt.savefig("accuracy_vs_severity.png", dpi=150)
plt.show()

## Step 8: Inference latency

How long does one forward pass take? Details that matter for a fair measurement:

- **Warm-up:** the first few runs are slower (GPU kernels get set up), so we run some untimed iterations first.
- **Synchronize:** GPU work is asynchronous. Without `torch.cuda.synchronize()` the clock would stop before the GPU finishes and you would time almost nothing.
- **Repeat:** we time many runs and report mean, standard deviation and median.
- **Batch size 1 vs. 32:** batch size 1 mimics a single request; larger batches show throughput.
- We time only the model's forward pass on random input in standard 32-bit precision. It does **not** include image loading, resizing, or degradation. Results depend on the hardware Colab assigns you, so always report the GPU/CPU model alongside the numbers.

In [ ]:
@torch.inference_mode()
def measure_latency(model, batch_size, dev, n_warmup, n_runs):
    model = model.to(dev).eval()
    x = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=dev)   # content does not affect speed
    for _ in range(n_warmup):                                         # warm-up (not timed)
        model(x)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    times_ms = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model(x)
        if dev.type == "cuda":
            torch.cuda.synchronize()                                  # wait for the GPU to finish
        times_ms.append((time.perf_counter() - t0) * 1000)
    t = np.array(times_ms)
    return t.mean(), t.std(), np.median(t)


latency_rows = []
devices_to_test = [device] if device.type == "cpu" else [device, torch.device("cpu")]

for dev in devices_to_test:
    n_warmup, n_runs = (10, 100) if dev.type == "cuda" else (3, 20)   # CPU is slow, so fewer runs
    hw = torch.cuda.get_device_name(0) if dev.type == "cuda" else f"CPU ({torch.get_num_threads()} threads)"
    for name, model in trained_models.items():
        model_copy = copy.deepcopy(model)                             # so we never disturb the trained models
        for bs in [1, 32]:
            mean_ms, std_ms, med_ms = measure_latency(model_copy, bs, dev, n_warmup, n_runs)
            latency_rows.append(dict(model=name, hardware=hw, batch_size=bs,
                                     params=count_params(model),
                                     mean_ms_per_batch=mean_ms, std_ms=std_ms, median_ms=med_ms,
                                     mean_ms_per_image=mean_ms / bs))
            print(f"{name:10s} | {hw} | batch {bs:2d} | {mean_ms:.2f} +/- {std_ms:.2f} ms per batch")

latency = pd.DataFrame(latency_rows)
latency.to_csv("results_latency.csv", index=False)
display(latency.round(3))

## Step 9: Save your outputs

Files created in this session: `results_accuracy.csv`, `results_latency.csv`, `accuracy_vs_severity.png`, `degradation_examples.png`.
Colab deletes them when the session ends, so download them now.

In [ ]:
try:
    from google.colab import files
    for f in ["results_accuracy.csv", "results_latency.csv", "accuracy_vs_severity.png", "degradation_examples.png"]:
        files.download(f)
except Exception as e:
    print("Download them manually from the Files panel on the left. (", e, ")")

## How to write this up honestly (useful for an MSc application)

**Report what you actually ran:** dataset, number of epochs, learning rate, number of evaluation images, GPU model, library versions (printed at the top), and the fact that severity levels are self-defined.

**Be careful with conclusions.** This is a single run with one random seed and a limited number of test images. Look at the error bars before claiming that one model is "more robust". Also remember the two models differ in size and in pretraining details, so any difference cannot be attributed purely to "transformer vs. CNN".

**Easy, meaningful extensions** (each one makes the project stronger):
1. Repeat the whole experiment with 3 or more seeds and report mean and standard deviation.
2. Use the full test set (`NUM_EVAL_IMAGES = len(test_raw)`) for tighter error bars.
3. Add more degradations (JPEG compression, rain, motion blur) or combine two at once.
4. Try training with some degraded images (data augmentation) and see whether robustness improves.
5. Compare a larger pair of models, or look at accuracy *and* latency together (a speed-vs-robustness trade-off plot).
6. Look at *where* each model fails (confusion matrices, or which classes break first).